In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [2]:
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

2026-03-06 10:57:40.631126: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 10:57:40.688805: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 10:57:42.138152: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
model = load_model("fridge_nilm_lstm.keras")
print("Model loaded successfully")

Model loaded successfully


2026-03-06 10:57:45.068072: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-03-06 10:57:45.068129: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2026-03-06 10:57:45.068137: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2026-03-06 10:57:45.068143: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-03-06 10:57:45.068149: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: vega2
2026-03-06 10:57:45.068153: I external/local_xla/xla/stream_executor/cuda/cuda_di

In [4]:
import joblib

fridge_scaler = joblib.load("fridge_scaler.pkl")
mains_scaler = joblib.load("mains_scaler.pkl")

In [5]:
WINDOW_SIZE = 60

In [6]:
df = pd.read_csv("last_1day_data.csv")

mains_power = df["active_power"].values.reshape(-1,1)

print("Total samples:", len(mains_power))

Total samples: 1441


In [7]:
mains_scaled = mains_scaler.transform(mains_power)

/home/jasil/miniconda3/envs/nilm-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [8]:
X_input = []

for i in range(len(mains_scaled) - WINDOW_SIZE):
    window = mains_scaled[i:i+WINDOW_SIZE]
    X_input.append(window)

X_input = np.array(X_input)

print("Input shape:", X_input.shape)

Input shape: (1381, 60, 1)


In [9]:
y_pred_norm = model.predict(X_input)

44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [10]:
y_pred_watts = fridge_scaler.inverse_transform(y_pred_norm)

In [11]:
FRIDGE_THRESHOLD = 20

y_pred_final = np.where(y_pred_watts < FRIDGE_THRESHOLD, 0, y_pred_watts)

In [14]:
fridge_prediction = np.zeros(len(mains_power))

fridge_prediction[WINDOW_SIZE:] = y_pred_final.flatten()

In [17]:
print(fridge_prediction[100:150])

[75.56178284 75.9368515  73.43235779 70.44100189 67.10314941 68.06121826
 68.13304138 65.8553009  62.97794724 61.04236221 62.57260513 66.89360046
 71.14572906 74.36436462 76.63054657 78.51847839 78.70509338 77.71778107
 74.39745331 69.95738983 66.92050171 65.84457397 65.67554474 64.09633636
 62.9916954  64.60018158 67.90241241 71.33841705 73.4444809  75.22618103
 77.70870209 79.86361694 81.09263611 80.54463959 79.78121185 79.72637177
 79.66165924 79.18240356 77.77789307 76.66573334 76.63719177 76.81368256
 76.88600159 76.9772644  77.05076599 77.37832642 77.03502655 76.99031067
 77.8521347  78.91788483]
